# Full model: from prices to the simulation of monthly contributions

**This notebook is self-contained:** all the model code (features, quantile model, prediction, simulation and validation)
is inside it. The only thing it uses from the repo is `data/fetch.py`, to download prices.

| Part | What it contains |
|---|---|
| **A. Model code** | The functions and classes (they only run once to define them) |
| **B. Using it** | Data, features, training, prediction, validation, contribution simulation and risk capacity |

Requirements: `pip install numpy pandas scikit-learn matplotlib yfinance requests`

> Educational prototype. Not financial advice.
> This notebook is for exploring. The app uses the same code from `model/`: if you change something here, change it there too.

Change the parameters in the next cell and run everything (`Run All`).

In [ ]:
# --- Parameters ---------------------------------------------------------
ACTIVO = "SPY"          # any symbol from data/assets.py
PLAZO_DIAS = 90         # horizon in calendar days (for section 5)
MONTO_TOTAL = 12000     # how much they want to invest in total
MESES = 12              # over how many months it is split (one part each month)
SUELDO_MENSUAL = 2500   # only for section 8 (risk capacity)
FUENTE = "real"         # "real" = Yahoo/Binance via data.fetch
                        # "sintetico" = made-up prices, ONLY for testing without internet

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from sklearn.ensemble import HistGradientBoostingRegressor

# Only access to the repo: data/fetch.py (cached prices). Searched upwards from this folder.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "fetch.py").exists()), None)
if ROOT is not None:
    sys.path.insert(0, str(ROOT))
try:
    from data.assets import all_symbols, label
    from data.fetch import get_close
except ImportError:
    all_symbols = get_close = None
    label = lambda s: s
    if FUENTE == "real":
        raise ImportError("For FUENTE = 'real' open this notebook inside the OxHack2026 repo (it needs data/fetch.py).")

# Chart style: light background, thin lines, subtle grid, no dual axis.
C1, C2, C3, GRIS = "#2a78d6", "#eb6834", "#1baf7a", "#8a8984"
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"
mpl.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "lines.linewidth": 2, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "figure.dpi": 110,
})
pct = FuncFormatter(lambda v, _: f"{v*100:.0f}%")
dinero = FuncFormatter(lambda v, _: f"{v:,.0f}")


# Part A. Model code

Each cell defines functions; it prints nothing. Run them in order (or `Run All`).



## A1. Features and target

Everything is **causal**: a day's feature only uses prices from that day or earlier. Features are normalised by the
asset's volatility, so a single model works for every asset. Time unit: "rows" of the series
(for stocks, one market day; for crypto, one calendar day). `plazo_a_filas` converts calendar days into rows.


In [ ]:
from __future__ import annotations

EWMA_LAMBDA = 0.94


HORIZONS = (21, 63, 126, 252)


MOMENTUM_WINDOWS = (5, 21, 63, 126, 252)


MIN_HISTORY = 300


FEATURE_COLS = [
    *[f"ret_{k}" for k in MOMENTUM_WINDOWS],
    "vol_ratio_21_252",
    "vol_ratio_63_252",
    "vol_ratio_ewma_252",
    "dist_ma200",
    "drawdown_252",
    "log_vol_anual",
]


MODEL_COLS = FEATURE_COLS + ["log_h"]


def filas_por_anio(index: pd.DatetimeIndex) -> float:
    """How many observations there are per year (≈252 for stocks, ≈365 for crypto)."""
    last = index[-1]
    n = int(((index > last - pd.Timedelta(days=365)) & (index <= last)).sum())
    return float(min(max(n, 200), 366))


def plazo_a_filas(plazo_dias: int, filas_anio: float) -> int:
    """Converts a horizon in calendar days into rows of the series."""
    return max(1, int(round(plazo_dias * filas_anio / 365.0)))


def log_returns(precios: pd.Series) -> pd.Series:
    return np.log(precios.astype(float)).diff()


def ewma_vol(r: pd.Series, lam: float = EWMA_LAMBDA) -> pd.Series:
    """Per-row volatility, RiskMetrics style."""
    return np.sqrt((r**2).ewm(alpha=1 - lam, adjust=False, min_periods=30).mean())


def build_features(precios: pd.Series) -> pd.DataFrame:
    """One row of features per date, plus the `sigma` column (EWMA vol per row)."""
    p = precios.astype(float).dropna()
    p = p[p > 0]
    logp = np.log(p)
    r = logp.diff()
    sigma = ewma_vol(r)
    out = pd.DataFrame(index=p.index)
    for k in MOMENTUM_WINDOWS:
        out[f"ret_{k}"] = (logp - logp.shift(k)) / (sigma * np.sqrt(k))
    vol21 = r.rolling(21).std()
    vol63 = r.rolling(63).std()
    vol252 = r.rolling(252, min_periods=200).std()
    out["vol_ratio_21_252"] = np.log(vol21 / vol252)
    out["vol_ratio_63_252"] = np.log(vol63 / vol252)
    out["vol_ratio_ewma_252"] = np.log(sigma / vol252)
    ma200 = logp.rolling(200).mean()
    out["dist_ma200"] = (logp - ma200) / (sigma * np.sqrt(20))
    out["drawdown_252"] = (logp - logp.rolling(252, min_periods=200).max()) / (sigma * np.sqrt(63))
    out["log_vol_anual"] = np.log(vol252 * np.sqrt(filas_por_anio(p.index)))
    out["sigma"] = sigma
    return out.replace([np.inf, -np.inf], np.nan)


def forward_log_return(precios: pd.Series, h: int) -> pd.Series:
    """log(P[t+h] / P[t]). NaN in the last h rows."""
    logp = np.log(precios.astype(float))
    return logp.shift(-h) - logp


def build_panel(series: dict[str, pd.Series], horizons=HORIZONS, step: int = 5) -> pd.DataFrame:
    """Training table with every asset and every horizon.

    Columns: MODEL_COLS, z (normalised target), fwd (actual log return),
    sigma, h, symbol, date and end_date (date on which the target becomes known; used
    to avoid mixing future and past when validating).
    step=5 takes one row out of every 5, to reduce overlap between targets.
    """
    frames = []
    for sym, precios in series.items():
        precios = precios.dropna()
        if len(precios) < MIN_HISTORY + 50:
            continue
        feats = build_features(precios)
        dates = pd.Series(precios.index, index=precios.index)
        for h in horizons:
            fwd = forward_log_return(precios, h)
            df = feats.copy()
            df["fwd"] = fwd
            df["z"] = fwd / (feats["sigma"] * np.sqrt(h))
            df["h"] = h
            df["log_h"] = np.log(h)
            df["symbol"] = sym
            df["date"] = df.index
            df["end_date"] = dates.shift(-h)
            df = df.dropna(subset=FEATURE_COLS + ["z", "end_date"])
            frames.append(df.iloc[::step])
    if not frames:
        raise ValueError("No asset has enough history to train.")
    return pd.concat(frames, ignore_index=True)


## A2. The quantile model

Gradient boosting with shallow depth and strong regularisation, **one model per quantile**. It predicts quantiles of the
normalised return `z = return / (volatility × √horizon)`. With several quantiles the full distribution is rebuilt
without assuming it is normal (`interp_quantiles`).


In [ ]:
from __future__ import annotations

QUANTILES = (0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99)


class QuantileModel:
    """Boosting with shallow depth and strong regularisation: prices are noise."""

    def __init__(self, max_iter: int = 100, learning_rate: float = 0.05,
                 max_depth: int = 3, min_samples_leaf: int = 300,
                 l2_regularization: float = 1.0, seed: int = 0):
        self.params = dict(max_iter=max_iter, learning_rate=learning_rate,
                           max_depth=max_depth, min_samples_leaf=min_samples_leaf,
                           l2_regularization=l2_regularization)
        self.seed = seed
        self.levels = np.array(QUANTILES)
        self.models: list[HistGradientBoostingRegressor] = []

    def fit(self, X: pd.DataFrame, z: np.ndarray) -> "QuantileModel":
        X = X[MODEL_COLS]
        self.models = []
        for q in self.levels:
            m = HistGradientBoostingRegressor(
                loss="quantile", quantile=float(q), early_stopping=False,
                random_state=self.seed, **self.params)
            self.models.append(m.fit(X, z))
        return self

    def predict_z(self, X: pd.DataFrame) -> np.ndarray:
        """Quantiles of z, shape (n, n_quantiles), sorted (no crossings)."""
        X = X[MODEL_COLS]
        out = np.column_stack([m.predict(X) for m in self.models])
        return np.sort(out, axis=1)


def interp_quantiles(levels: np.ndarray, values: np.ndarray, u: np.ndarray) -> np.ndarray:
    """Quantile function by linear interpolation; the tails are extended with the
    slope of the last segment (so extreme cases are not clipped)."""
    levels = np.asarray(levels, dtype=float)
    values = np.asarray(values, dtype=float)
    u = np.asarray(u, dtype=float)
    out = np.interp(u, levels, values)
    lo, hi = u < levels[0], u > levels[-1]
    if lo.any():
        slope = (values[1] - values[0]) / (levels[1] - levels[0])
        out[lo] = values[0] + slope * (u[lo] - levels[0])
    if hi.any():
        slope = (values[-1] - values[-2]) / (levels[-1] - levels[-2])
        out[hi] = values[-1] + slope * (u[hi] - levels[-1])
    return out


## A3. Load data and train


In [ ]:
from __future__ import annotations

def load_series(symbols: list[str] | None = None, verbose: bool = True) -> dict[str, pd.Series]:
    series = {}
    for s in symbols or all_symbols():
        try:
            series[s] = get_close(s)
        except Exception as e:  # one failing asset must not break training
            if verbose:
                print(f"  (skipping {s}: {e})")
    return series


def train(series: dict[str, pd.Series], step: int = 5) -> QuantileModel:
    panel = build_panel(series, HORIZONS, step=step)
    return QuantileModel().fit(panel[MODEL_COLS], panel["z"].to_numpy())


## A4. Prediction (the contract with the rest of the team)

`predict(precios, plazo_dias)` returns `rendimiento_esperado`, `volatilidad`, `prob_perdida` and `intervalo` at 90 %
(simple returns: 0.012 = +1.2 %). The model is trained with horizons of 21, 63, 126 and 252 rows; for any other
horizon it interpolates between the two closest ones, to avoid producing steps.


In [ ]:
from __future__ import annotations

_U = (np.arange(2000) + 0.5) / 2000


def predict_distribution(precios: pd.Series, plazo_dias: int,
                         model: QuantileModel | None = None) -> tuple[np.ndarray, np.ndarray]:
    """Quantiles of the simple return at `plazo_dias`: (levels, values)."""
    if model is None:
        raise ValueError("Pass the trained model: predict(..., model=model)")
    precios = precios.dropna()
    if len(precios) < MIN_HISTORY:
        raise ValueError(f"Not enough history: {len(precios)} rows, {MIN_HISTORY} are needed.")
    feats = build_features(precios).iloc[[-1]]
    if feats.drop(columns="sigma").isna().any(axis=None):
        raise ValueError("The features for the last day cannot be computed (incomplete data).")
    h = plazo_a_filas(plazo_dias, filas_por_anio(precios.index))
    z = _z_interpolado(model, feats.drop(columns="sigma"), h)
    log_ret = z * float(feats["sigma"].iloc[0]) * np.sqrt(h)
    return model.levels, np.expm1(log_ret)


def _z_interpolado(model: QuantileModel, feats: pd.DataFrame, h: int) -> np.ndarray:
    """Quantiles of z for a horizon of h rows.

    The model is only trained with the HORIZONS horizons. Between them, the trees
    would give steps, so it predicts at the two closest trained horizons
    and interpolates in log(h). Outside the range the extreme horizon is used (scaling
    by sqrt(h) already adjusts the magnitude).
    """
    hs = HORIZONS
    if h <= hs[0]:
        lo = hi = hs[0]
    elif h >= hs[-1]:
        lo = hi = hs[-1]
    else:
        hi = next(x for x in hs if x >= h)
        lo = max(x for x in hs if x <= h)
    zs = {k: model.predict_z(feats.assign(log_h=np.log(k)))[0] for k in {lo, hi}}
    if lo == hi:
        return zs[lo]
    w = (np.log(h) - np.log(lo)) / (np.log(hi) - np.log(lo))
    return (1 - w) * zs[lo] + w * zs[hi]


def summarize(levels: np.ndarray, values: np.ndarray) -> dict:
    s = interp_quantiles(levels, values, _U)
    lo, hi = interp_quantiles(levels, values, np.array([0.05, 0.95]))
    return {
        "rendimiento_esperado": float(s.mean()),
        "volatilidad": float(s.std()),
        "prob_perdida": float((s < 0).mean()),
        "intervalo": (float(lo), float(hi)),
    }


def predict(precios: pd.Series, plazo_dias: int, model: QuantileModel | None = None) -> dict:
    levels, values = predict_distribution(precios, plazo_dias, model)
    return summarize(levels, values)


## A5. Simulation of monthly contributions

Splits the amount into equal parts, one at the start of each month, and draws 5,000 futures with the 1-month distribution.
Limitation: months are drawn independently and with today's distribution.


In [ ]:
from __future__ import annotations

DIAS_POR_MES = 30


def simulate_dca(precios: pd.Series, monto_total: float, meses: int,
                 n_sims: int = 5000, seed: int = 0,
                 model: QuantileModel | None = None) -> dict:
    if meses < 1 or monto_total <= 0:
        raise ValueError("meses must be >= 1 and monto_total > 0.")
    levels, values = predict_distribution(precios, DIAS_POR_MES, model)
    rng = np.random.default_rng(seed)
    # Monthly simple returns; never below -100 %.
    R = np.maximum(interp_quantiles(levels, values, rng.random((n_sims, meses))), -0.99)

    aporte = monto_total / meses
    dca = np.zeros((n_sims, meses + 1))
    for k in range(meses):
        dca[:, k + 1] = (dca[:, k] + aporte) * (1 + R[:, k])
    lump = np.column_stack([np.full(n_sims, monto_total), monto_total * np.cumprod(1 + R, axis=1)])

    aportado = aporte * np.minimum(np.arange(meses + 1), meses)
    final_dca, final_lump = dca[:, -1], lump[:, -1]
    pct = lambda a: {f"p{p}": float(np.percentile(a, p)) for p in (5, 25, 50, 75, 95)}  # noqa: E731
    return {
        "aporte_mensual": float(aporte),
        "total_aportado": float(monto_total),
        "valor_final_dca": pct(final_dca),
        "valor_final_todo_junto": pct(final_lump),
        "prob_perder_dinero_dca": float((final_dca < monto_total).mean()),
        "prob_perder_dinero_todo_junto": float((final_lump < monto_total).mean()),
        "prob_dca_mejor_que_todo_junto": float((final_dca > final_lump).mean()),
        "peor_caso_razonable_dca": float(np.percentile(final_dca, 5) - monto_total),  # p5, in money
        "trayectoria_dca": {  # for plotting: portfolio value month by month
            "meses": list(range(meses + 1)),
            "aportado": aportado.tolist(),
            "p5": np.percentile(dca, 5, axis=0).tolist(),
            "p50": np.percentile(dca, 50, axis=0).tolist(),
            "p95": np.percentile(dca, 95, axis=0).tolist(),
        },
    }


## A6. Walk-forward validation

For each test year, the model is trained only on examples whose outcome was already known before that year
(no future leakage) and compared with a historical baseline of the asset itself.


In [ ]:
from __future__ import annotations

def pinball(y: np.ndarray, q: np.ndarray, levels: np.ndarray) -> float:
    """Mean pinball loss. y: (n,), q: (n, k) or (k,)."""
    q = np.broadcast_to(q, (len(y), len(levels)))
    d = y[:, None] - q
    return float(np.mean(np.maximum(levels * d, (levels - 1) * d)))


def evaluate(series: dict[str, pd.Series], target: str, test_years: int = 5,
             horizons=HORIZONS, step: int = 5) -> pd.DataFrame:
    panel = build_panel(series, horizons, step=step)
    levels = np.array(QUANTILES)
    i05, i50, i95 = (QUANTILES.index(q) for q in (0.05, 0.50, 0.95))
    last_year = panel.loc[panel.symbol == target, "date"].max().year
    years = [y for y in range(last_year - test_years + 1, last_year + 1)]

    rows = {h: {"y": [], "q": [], "base_q": [], "base_mean": [], "p_loss": [], "base_p": []} for h in horizons}
    for y in years:
        start = pd.Timestamp(f"{y}-01-01")
        end = pd.Timestamp(f"{y + 1}-01-01")
        train = panel[panel.end_date < start]
        test = panel[(panel.symbol == target) & (panel.date >= start) & (panel.date < end)]
        if len(train) < 1000 or test.empty:
            continue
        model = QuantileModel().fit(train[MODEL_COLS], train["z"].to_numpy())
        z_hat = model.predict_z(test[MODEL_COLS])
        scale = (test["sigma"] * np.sqrt(test["h"])).to_numpy()
        q_hat = z_hat * scale[:, None]  # log-return quantiles
        for h in horizons:
            m = (test["h"] == h).to_numpy()
            hist = train[(train.symbol == target) & (train.h == h)]["fwd"].to_numpy()
            if m.sum() == 0 or len(hist) < 50:
                continue
            r = rows[h]
            r["y"].append(test["fwd"].to_numpy()[m])
            r["q"].append(q_hat[m])
            r["base_q"].append(np.tile(np.quantile(hist, levels), (m.sum(), 1)))
            r["base_mean"].append(np.full(m.sum(), hist.mean()))
            # Model's probability of loss: level where the quantile function crosses 0.
            p = np.array([np.interp(0.0, qq, levels, left=0.0005, right=0.9995) for qq in q_hat[m]])
            r["p_loss"].append(p)
            r["base_p"].append(np.full(m.sum(), (hist < 0).mean()))

    out = []
    for h in horizons:
        r = rows[h]
        if not r["y"]:
            continue
        y = np.concatenate(r["y"]); q = np.vstack(r["q"]); bq = np.vstack(r["base_q"])
        bm = np.concatenate(r["base_mean"]); p = np.concatenate(r["p_loss"]); bp = np.concatenate(r["base_p"])
        loss = (y < 0).astype(float)
        pin_m, pin_b = pinball(y, q, levels), pinball(y, bq, levels)
        out.append({
            "plazo_filas": h, "n": len(y),
            "pinball_modelo": pin_m, "pinball_base": pin_b, "skill_pinball": 1 - pin_m / pin_b,
            "cobertura90_modelo": float(((y >= q[:, i05]) & (y <= q[:, i95])).mean()),
            "cobertura90_base": float(((y >= bq[:, i05]) & (y <= bq[:, i95])).mean()),
            "mae_mediana_modelo": float(np.mean(np.abs(y - q[:, i50]))),
            "mae_cero": float(np.mean(np.abs(y))),
            "mae_media_hist": float(np.mean(np.abs(y - bm))),
            "brier_modelo": float(np.mean((p - loss) ** 2)),
            "brier_base": float(np.mean((bp - loss) ** 2)),
        })
    return pd.DataFrame(out)


## A7. Synthetic prices (only for testing without internet)

Used only if `FUENTE = "sintetico"`. They simulate clustered volatility and heavy tails. **They are not real data.**


In [ ]:
def fake_prices(n=2500, seed=0, mu=0.0003, end="2026-10-23"):
    """Returns with clustered volatility (GARCH-like) and heavy tails."""
    rng = np.random.default_rng(seed)
    var, r = 0.0001, np.empty(n)
    for i in range(n):
        r[i] = mu + np.sqrt(var) * rng.standard_t(5) * np.sqrt(3 / 5)
        var = 0.000002 + 0.08 * r[i] ** 2 + 0.90 * var
    idx = pd.bdate_range(end=end, periods=n)
    return pd.Series(100 * np.exp(np.cumsum(r)), index=idx, name=f"FAKE{seed}")


---
# Part B. Using it


## 1. Data

`load_series()` uses the cache in `data/cache/` and only downloads what is missing
(Yahoo Finance for stocks, ETFs, indices and FX; Binance for crypto). If an asset fails, it is skipped.
The model is trained with **all** assets at once; that is why it is worth having the full catalogue.

In [ ]:
if FUENTE == "real":
    t0 = time.time()
    series = load_series()
    print(f"Download/read: {time.time() - t0:.1f} s")
else:
    print("⚠️  FUENTE = 'sintetico': MADE-UP prices. Nothing that comes out here is real.")
    nombres = ["SPY", "QQQ", "GLD", "AAPL", "MSFT", "NVDA", "VOO", "BTCUSDT"]
    series = {s: fake_prices(n=5000, seed=i) for i, s in enumerate(nombres)}

assert ACTIVO in series, f"No data for {ACTIVO}. Run: python -m data.fetch {ACTIVO}"
precios = series[ACTIVO]

resumen = pd.DataFrame({
    "asset": [label(s) for s in series],
    "rows": [len(p) for p in series.values()],
    "from": [p.index.min().date() for p in series.values()],
    "to": [p.index.max().date() for p in series.values()],
    "last close": [round(float(p.iloc[-1]), 2) for p in series.values()],
}, index=list(series))
resumen

## 2. The chosen asset: price and volatility

**Volatility** (how strongly the price moves) can be predicted better than direction, because
turbulent periods cluster. Here it is shown with a weighted average that gives more weight to recent days (EWMA).

In [ ]:
vol_anual = ewma_vol(log_returns(precios)) * np.sqrt(filas_por_anio(precios.index))

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
ax[0].plot(precios.index, precios.values, color=C1)
ax[0].set_yscale("log")
ax[0].yaxis.set_major_formatter(dinero); ax[0].yaxis.set_minor_formatter(dinero)
ax[0].set_title(f"{label(ACTIVO)}: closing price (log scale)")
ax[1].plot(vol_anual.index, vol_anual.values, color=C2)
ax[1].yaxis.set_major_formatter(pct)
ax[1].set_title("Annualised volatility (EWMA)")
plt.tight_layout()
plt.show()

## 3. Features and target

For each day, features are computed that **only use the past**:

| Feature | What it measures |
|---|---|
| `ret_5 … ret_252` | Momentum: how much it rose or fell over the last 5 to 252 days |
| `vol_ratio_*` | Whether the asset is more or less nervous than usual |
| `dist_ma200` | Distance of the price to its 200-day average |
| `drawdown_252` | How far it is from its high of the last year |
| `log_vol_anual` | The asset's own level of volatility |

All are **divided by the asset's volatility**, so a single model works for Bitcoin, gold or the S&P 500.

**Target `z`** = future return / (volatility × √horizon). It is what the model learns to predict,
for each horizon in `HORIZONS` (21, 63, 126 and 252 rows ≈ 1, 3, 6 and 12 months).

In [ ]:
feats = build_features(precios)
display(feats[FEATURE_COLS].dropna().tail(5).round(3))

panel = build_panel(series, HORIZONS, step=5)
print(f"Training table: {len(panel):,} examples from {panel.symbol.nunique()} assets and {len(HORIZONS)} horizons")

fig, ax = plt.subplots(figsize=(10, 3.6))
bins = np.linspace(-4, 4, 81)
for h, c in [(21, C1), (252, C2)]:
    ax.hist(panel.loc[panel.h == h, "z"].clip(-4, 4), bins=bins, density=True, histtype="stepfilled",
            color=c, alpha=0.35, edgecolor=c, linewidth=1.5, label=f"horizon {h} rows")
ax.set_title("Target z (volatility-normalised return): all assets")
ax.set_xlabel("z"); ax.set_yticks([])
ax.legend(frameon=False)
plt.show()

## 4. Train the model

A **quantile gradient boosting** (scikit-learn): 9 small models, one per quantile
(1, 5, 10, 25, 50, 75, 90, 95 and 99 %). Shallow depth and strong regularisation, because prices are very noisy
and a large model would only memorise noise.

This training uses the **whole** history, and it is the one used to predict today.
Section 6 validates the same method with trainings that only see the past.

In [ ]:
t0 = time.time()
model = train(series)
print(f"Model trained in {time.time() - t0:.1f} s with {len(series)} assets")

## 5. What the model predicts

**It does not predict a price.** It predicts the **distribution** of the return over a horizon: how likely
each outcome is, from very bad to very good. From it come the four numbers of the `predict()` contract:
expected return, volatility, probability of loss and 90 % interval.

In [ ]:
pred = predict(precios, PLAZO_DIAS, model=model)
lo, hi = pred["intervalo"]
print(f"{label(ACTIVO)} over {PLAZO_DIAS} days")
print(f"  Expected return      : {pred['rendimiento_esperado']:+.1%}")
print(f"  Volatility           : {pred['volatilidad']:.1%}")
print(f"  Probability of loss  : {pred['prob_perdida']:.0%}")
print(f"  90 % interval        : from {lo:+.1%} to {hi:+.1%}")

In [ ]:
levels, vals = predict_distribution(precios, PLAZO_DIAS, model)
u = (np.arange(20000) + 0.5) / 20000
muestras = interp_quantiles(levels, vals, u)

# Smoothed density, without the extreme tails (0.5 % on each side) so it reads well.
a, b = np.quantile(muestras, [0.005, 0.995])
conteo, bordes = np.histogram(muestras, bins=90, range=(a, b), density=True)
kernel = np.exp(-0.5 * (np.arange(-6, 7) / 2.5) ** 2); kernel /= kernel.sum()
dens = np.convolve(conteo, kernel, mode="same")
centros = (bordes[:-1] + bordes[1:]) / 2
ancho = bordes[1] - bordes[0]

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.bar(centros, dens, width=ancho * 0.95, color=np.where(centros < 0, C2, C1))
mediana = np.median(muestras)
ax.axvline(0, color=INK2, lw=1)
ax.axvline(mediana, color=INK, lw=1.5, ls="--")
for x in (lo, hi):
    ax.axvline(x, color=INK2, lw=1, ls=":")
ax.set_ylim(0, dens.max() * 1.3)
top = ax.get_ylim()[1]
ax.text(mediana, top * 0.80, " median", color=INK, va="top", fontsize=9)
ax.text(lo, top * 0.80, "5 % ", color=INK2, ha="right", va="top", fontsize=9)
ax.text(hi, top * 0.80, " 95 %", color=INK2, ha="left", va="top", fontsize=9)
gap = (b - a) * 0.01
ax.text(-gap, top * 0.97, f"loss {pred['prob_perdida']:.0%}", color=INK, ha="right", va="top", fontsize=11, fontweight="bold")
ax.text(gap, top * 0.97, f"gain {1 - pred['prob_perdida']:.0%}", color=INK, ha="left", va="top", fontsize=11, fontweight="bold")
ax.xaxis.set_major_formatter(pct); ax.set_yticks([])
ax.set_title(f"{label(ACTIVO)}: possible outcomes over {PLAZO_DIAS} days")
plt.show()

**How uncertainty changes with the horizon.** The longer the horizon, the wider the range: the model does not promise more return,
it only says there is more room for surprises, good and bad.

In [ ]:
plazos = list(range(30, 361, 30))
idx = {q: QUANTILES.index(q) for q in (0.05, 0.25, 0.50, 0.75, 0.95)}
Q = np.array([predict_distribution(precios, p, model)[1] for p in plazos])

fig, ax = plt.subplots(figsize=(10, 4.2))
meses_x = np.array(plazos) / 30
ax.fill_between(meses_x, Q[:, idx[0.05]], Q[:, idx[0.95]], color=C1, alpha=0.18, lw=0)
ax.fill_between(meses_x, Q[:, idx[0.25]], Q[:, idx[0.75]], color=C1, alpha=0.35, lw=0)
ax.plot(meses_x, Q[:, idx[0.50]], color=C1)
ax.axhline(0, color=INK2, lw=1)
ax.text(meses_x[-1] + 0.15, Q[-1, idx[0.95]], "90 % of cases", color=INK2, va="center", fontsize=9)
ax.text(meses_x[-1] + 0.15, Q[-1, idx[0.75]], "50 % of cases", color=INK2, va="center", fontsize=9)
ax.text(meses_x[-1] + 0.15, Q[-1, idx[0.50]], "median", color=INK2, va="center", fontsize=9)
ax.set_xlim(1, meses_x[-1] + 2.2)
ax.yaxis.set_major_formatter(pct); ax.set_xlabel("months")
ax.set_title(f"{label(ACTIVO)}: return range by horizon")
plt.show()

tabla = pd.DataFrame([{"horizon (days)": p, **{k: v for k, v in predict(precios, p, model=model).items()}} for p in (30, 90, 180, 365)])
tabla["intervalo"] = tabla["intervalo"].map(lambda t: f"{t[0]:+.1%} to {t[1]:+.1%}")
for c in ("rendimiento_esperado", "volatilidad"):
    tabla[c] = tabla[c].map("{:+.1%}".format if c == "rendimiento_esperado" else "{:.1%}".format)
tabla["prob_perdida"] = tabla["prob_perdida"].map("{:.0%}".format)
tabla

## 6. Validation: does it beat a baseline?

**Walk-forward:** for each of the last years, the model is trained only on examples whose outcome was already
known before that year, and tested on that year. Dates are never shuffled at random.

**Baseline:** the historical distribution of the asset itself (its quantiles, its mean and its share of losses in the past).
It is what someone without a model would do.

| Metric | How to read it |
|---|---|
| `skill_pinball` | Improvement over the baseline in quantile error. **> 0 = better than the baseline** |
| `cobertura90` | The 5–95 % interval should contain about **90 %** of actual cases |
| `mae_*` | Error of the median vs. predicting zero return or the historical mean (lower is better) |
| `brier_*` | Quality of the probability of loss (lower is better) |

It may take a couple of minutes.

In [ ]:
t0 = time.time()
res = evaluate(series, ACTIVO, test_years=5)
print(f"Evaluation: {time.time() - t0:.0f} s")
res.round(4)

In [ ]:
etiqueta = [f"{int(h)} rows\n(~{round(h / 21)} m)" for h in res["plazo_filas"]]
x = np.arange(len(res)); w = 0.38

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
colores = [C1 if v >= 0 else C2 for v in res["skill_pinball"]]
ax[0].bar(x, res["skill_pinball"], color=colores, width=0.6)
ax[0].axhline(0, color=INK2, lw=1)
ax[0].set_xticks(x, etiqueta); ax[0].yaxis.set_major_formatter(pct)
ax[0].set_title("Improvement over the baseline\n(blue = better, orange = worse)")
for xi, v in zip(x, res["skill_pinball"]):
    ax[0].text(xi, v, f"{v:+.1%}", ha="center", va="bottom" if v >= 0 else "top", fontsize=9)

ax[1].bar(x - w / 2, res["cobertura90_modelo"], w, color=C1, label="model")
ax[1].bar(x + w / 2, res["cobertura90_base"], w, color=GRIS, label="baseline")
ax[1].axhline(0.9, color=INK, lw=1.2, ls="--")
ax[1].text(len(res) - 0.55, 0.915, "ideal 90 %", ha="right", va="bottom", fontsize=9, color=INK)
ax[1].set_xticks(x, etiqueta); ax[1].set_ylim(0, 1.2); ax[1].yaxis.set_major_formatter(pct)
ax[1].set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax[1].set_title("Coverage of the 90 % interval")
ax[1].legend(frameon=False, loc="upper left", ncol=2)
plt.tight_layout(); plt.show()

mejora = int((res["skill_pinball"] > 0).sum())
print(f"The model beats the baseline in {mejora} of {len(res)} horizons.")
print(f"Mean coverage of the 90 % interval: model {res['cobertura90_modelo'].mean():.0%}, baseline {res['cobertura90_base'].mean():.0%}.")

**How to interpret it.** With prices, beating the baseline by a lot is rare. An honest result is:
a small but positive improvement, and an interval whose coverage gets close to 90 %. If coverage is far off,
the regularisation of `QuantileModel` is adjusted. The numbers in this section are the ones that go into the README.

## 7. Simulate investing a part each month

`simulate_dca` splits the amount into equal parts, one at the start of each month, and draws 5,000 possible futures using
the **1-month** distribution predicted by the model. It compares against investing everything on the first day **with the same
drawn returns**, so the comparison is fair.

In [ ]:
sim = simulate_dca(precios, MONTO_TOTAL, MESES, n_sims=5000, model=model)
d, t = sim["valor_final_dca"], sim["valor_final_todo_junto"]
print(f"{label(ACTIVO)}: {MONTO_TOTAL:,.0f} over {MESES} months = {sim['aporte_mensual']:,.0f} per month")
print(f"  Final value, in parts     : typical {d['p50']:,.0f}   (90 % range: {d['p5']:,.0f} to {d['p95']:,.0f})")
print(f"  Final value, all at once  : typical {t['p50']:,.0f}   (90 % range: {t['p5']:,.0f} to {t['p95']:,.0f})")
print(f"  Prob. of ending with less than invested: in parts {sim['prob_perder_dinero_dca']:.0%}, all at once {sim['prob_perder_dinero_todo_junto']:.0%}")
print(f"  Prob. that investing in parts ends better than all at once: {sim['prob_dca_mejor_que_todo_junto']:.0%}")
print(f"  Reasonable worst case (5 %), in parts: {sim['peor_caso_razonable_dca']:+,.0f}")

In [ ]:
tr = sim["trayectoria_dca"]; m = tr["meses"]

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.2, 1]})

ax[0].fill_between(m, tr["p5"], tr["p95"], color=C1, alpha=0.2, lw=0)
ax[0].plot(m, tr["p50"], color=C1)
ax[0].plot(m, tr["aportado"], color=INK2, lw=1.5, ls="--")
etiquetas = sorted([(tr["p95"][-1], "best 5 %", INK2), (tr["p50"][-1], "median", C1),
                    (tr["aportado"][-1], "contributed", INK2), (tr["p5"][-1], "worst 5 %", INK2)])
sep = (tr["p95"][-1] - tr["p5"][-1]) * 0.17          # minimum spacing between labels
ys = [e[0] for e in etiquetas]
for i in range(1, len(ys)):
    ys[i] = max(ys[i], ys[i - 1] + sep)
for y, (_, txt, col) in zip(ys, etiquetas):
    ax[0].text(m[-1] + 0.2, y, txt, color=col, va="center", fontsize=9)
ax[0].set_xlim(0, m[-1] + 2.6); ax[0].yaxis.set_major_formatter(dinero)
ax[0].set_xlabel("month"); ax[0].set_title("Portfolio value month by month (investing in parts)")

filas = [("In parts\n(each month)", d, C1), ("All at\nonce", t, C2)]
for i, (nombre, v, c) in enumerate(filas):
    ax[1].hlines(i, v["p5"], v["p95"], color=c, lw=3, alpha=0.55)
    ax[1].hlines(i, v["p25"], v["p75"], color=c, lw=11)
    ax[1].plot(v["p50"], i, "o", ms=8, mfc=SURF, mec=INK, mew=1.5)
ax[1].axvline(MONTO_TOTAL, color=INK2, ls="--", lw=1.2)
ax[1].text(MONTO_TOTAL, 1.45, " amount invested", color=INK2, fontsize=9, va="bottom")
ax[1].set_yticks([0, 1], [f[0] for f in filas]); ax[1].set_ylim(-0.6, 1.8)
ax[1].xaxis.set_major_formatter(dinero); ax[1].grid(axis="y", visible=False)
ax[1].set_title("Final value: 90 % of cases (thin), 50 % (thick), median (dot)")
plt.tight_layout(); plt.show()

**How to read it.** With a positive expected return, investing everything from the start usually ends better *on average*,
because the money spends more time invested. Investing in parts reduces exposure at the beginning and softens
the blow if the market falls early. The chart shows that trade-off: more safety in exchange for a lower typical gain.

## 8. Preview: is the amount reasonable for this person?

The salary **does not enter the model**: it does not change how the asset moves, it changes how much damage a drop would do.
This is only a preview; the real rule will be written in `decision/`.
The thresholds are **examples** that the team must decide and justify.

In [ ]:
UMBRAL_APORTE = 0.30      # maximum monthly contribution as a fraction of salary (example)
UMBRAL_PERDIDA = 2.0      # bad-scenario loss, in months of salary (example)

aporte_vs_sueldo = sim["aporte_mensual"] / SUELDO_MENSUAL
perdida = max(0.0, -sim["peor_caso_razonable_dca"])
perdida_en_sueldos = perdida / SUELDO_MENSUAL
monto_max = MONTO_TOTAL * (UMBRAL_PERDIDA * SUELDO_MENSUAL) / perdida if perdida > 0 else None

print(f"Monthly contribution: {sim['aporte_mensual']:,.0f} = {aporte_vs_sueldo:.0%} of salary "
      f"({'⚠️ high' if aporte_vs_sueldo > UMBRAL_APORTE else 'reasonable'}; example threshold {UMBRAL_APORTE:.0%})")
print(f"Total amount: {MONTO_TOTAL / SUELDO_MENSUAL:.1f} monthly salaries")
print(f"Bad scenario (5 %): loss of {perdida:,.0f} = {perdida_en_sueldos:.1f} salaries "
      f"({'⚠️ high' if perdida_en_sueldos > UMBRAL_PERDIDA else 'tolerable'}; example threshold {UMBRAL_PERDIDA:.0f})")
if monto_max is not None:
    print(f"Maximum amount so the bad-scenario loss does not exceed {UMBRAL_PERDIDA:.0f} salaries: {monto_max:,.0f}")

## 9. What this model does and does not do

**Does:** estimates a range of outcomes with their probability, measures the risk of each horizon and simulates monthly contributions.

**Does not:**
- It does not predict the exact price nor get the direction right with certainty; its strength is measuring risk.
- It only sees price history: it knows nothing about news or events (Fed, Bank of England, ECB).
- The simulation draws months independently and with today's distribution.
- The salary does not say how much the person spends or whether they have savings.

**Next steps:** move the results of section 6 into the README, adjust the regularisation if coverage
is far from 90 %, and connect `predict()` and `simulate_dca()` with `decision/` and `app/`.